# Lab 16: DQN on CartPole

**Module 16** | Read `notes/16-dqn.pdf` before running this notebook.

Train a DQN agent with Stable-Baselines3 on CartPole-v1 and plot episode rewards.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Deep Q-Networks on CartPole

CartPole-v1 is a classic control benchmark: keep a pole balanced on a cart by pushing left or right.
The state is four continuous values (cart position, velocity, pole angle, angular velocity); the agent chooses one of two discrete actions.

**Deep Q-Networks (DQN)** approximate the action-value function $Q(s,a)$ with a neural network.
Experience replay and a target network stabilize training when bootstrapping from the network's own predictions.
Stable-Baselines3 wraps these details so you can focus on environment setup, monitoring, and evaluation.

In this lab you will:

1. Wrap the environment with `Monitor` so episode rewards are logged automatically.
2. Train a DQN policy for **30,000 environment steps** and print episode statistics during training.
3. Plot the reward curve and run **5 deterministic test episodes** to inspect actions and totals.
4. Compare your results to the CartPole **solved threshold** (average reward of 195 over 100 episodes).


### Environment setup

`Monitor` records every completed episode's total reward. We also register a lightweight callback that prints rolling stats whenever an episode finishes during training.


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

ROOT = Path("..").resolve()
ACTION_NAMES = {0: "left", 1: "right"}
SOLVED_THRESHOLD = 195


class EpisodeStatsCallback(BaseCallback):
    """Print episode reward and length as training progresses."""

    def __init__(self, print_every: int = 5):
        super().__init__()
        self.print_every = print_every
        self.episode_count = 0

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if info and "episode" in info:
                self.episode_count += 1
                ep = info["episode"]
                if self.episode_count % self.print_every == 0:
                    print(
                        f"  Training episode {self.episode_count}: "
                        f"reward={ep['r']:.0f}, length={ep['l']}"
                    )
        return True


env = Monitor(gym.make("CartPole-v1"))
obs_space = env.observation_space
act_space = env.action_space

print("CartPole-v1 environment ready")
print(f"  Observation space: {obs_space} (4 state variables)")
print(f"  Action space: {act_space} (0=left, 1=right)")
print(f"  Solved threshold: average reward >= {SOLVED_THRESHOLD} over 100 episodes")


### Train the DQN agent

`MlpPolicy` feeds the raw 4-dimensional state vector into a small MLP that outputs Q-values for both actions.
Training runs for 30,000 timesteps; watch the callback output to see rewards climb as the policy improves.


In [ ]:
TOTAL_TIMESTEPS = 30_000

model = DQN("MlpPolicy", env, verbose=1)
stats_callback = EpisodeStatsCallback(print_every=5)

print(f"Starting DQN training for {TOTAL_TIMESTEPS:,} timesteps...")
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=stats_callback)
print("Training finished.")


### Evaluate training and test the policy

First we summarize all logged training episodes and plot the reward curve.
Then we open a **fresh** test environment (no exploration noise) and run 5 greedy episodes, printing each action sequence and total reward.


In [ ]:
rewards = env.get_episode_rewards()
episode_lengths = env.get_episode_lengths()

print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Episodes completed during training: {len(rewards)}")
print(f"First episode reward: {rewards[0]:.0f}")
print(f"Last episode reward: {rewards[-1]:.0f}")
print(f"Max episode reward: {max(rewards):.0f}")
print(f"Overall mean reward: {np.mean(rewards):.2f}")
if len(rewards) >= 10:
    mean_last_10 = float(np.mean(rewards[-10:]))
    print(f"Mean reward (last 10 episodes): {mean_last_10:.2f}")
if len(rewards) >= 100:
    mean_last_100 = float(np.mean(rewards[-100:]))
    print(f"Mean reward (last 100 episodes): {mean_last_100:.2f}")
    solved = mean_last_100 >= SOLVED_THRESHOLD
    print(f"Solved (last 100 >= {SOLVED_THRESHOLD})? {solved}")
else:
    mean_last_10 = float(np.mean(rewards[-10:])) if len(rewards) >= 10 else float(np.mean(rewards))
    print(f"(Need 100 episodes for official solved check; using last 10 mean as proxy)")

plt.figure(figsize=(9, 4))
plt.plot(rewards, alpha=0.6, label="Episode reward")
if len(rewards) >= 10:
    window = np.convolve(rewards, np.ones(10) / 10, mode="valid")
    plt.plot(range(9, 9 + len(window)), window, linewidth=2, label="10-episode moving avg")
plt.axhline(SOLVED_THRESHOLD, color="gray", linestyle="--", label=f"solved threshold ({SOLVED_THRESHOLD})")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN on CartPole-v1: training rewards")
plt.legend()
plt.tight_layout()
plt.show()

print()
print("=" * 60)
print("5 DETERMINISTIC TEST EPISODES (greedy policy)")
print("=" * 60)

test_env = gym.make("CartPole-v1")
test_totals = []

for ep in range(5):
    obs, _ = test_env.reset()
    done = False
    step = 0
    total_reward = 0.0
    actions_taken: list[int] = []
    step_rewards: list[float] = []

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        action_int = int(action)
        actions_taken.append(action_int)
        obs, reward, terminated, truncated, _ = test_env.step(action_int)
        total_reward += reward
        step_rewards.append(float(reward))
        step += 1
        done = terminated or truncated

    test_totals.append(total_reward)
    action_str = ", ".join(ACTION_NAMES[a] for a in actions_taken[:12])
    if len(actions_taken) > 12:
        action_str += ", ..."
    print(f"Episode {ep + 1}:")
    print(f"  Steps: {step}")
    print(f"  Actions (first 12): {action_str}")
    print(f"  Per-step rewards (first 8): {step_rewards[:8]}")
    print(f"  Total reward: {total_reward:.0f}")

test_env.close()
env.close()

print()
print("=" * 60)
print("COMPARISON TO SOLVED THRESHOLD")
print("=" * 60)
print(f"Test episode mean reward: {np.mean(test_totals):.1f}")
print(f"CartPole solved criterion: average >= {SOLVED_THRESHOLD} over 100 consecutive episodes")
if len(rewards) >= 100:
    status = "SOLVED" if mean_last_100 >= SOLVED_THRESHOLD else "NOT YET SOLVED"
    print(f"Training status (last 100 episodes): {status}")
else:
    print("Train longer (more timesteps) if rewards have not plateaued near 500.")


## Interpretation

CartPole gives +1 reward per timestep until the pole falls (max 500 steps per episode).
A healthy learning curve climbs from near-random performance (often 20 to 40) toward the maximum.

If rewards plateau early, try more timesteps or tune hyperparameters (`learning_rate`, `buffer_size`, `exploration_fraction`).

**Key takeaway:** value-based RL with function approximation can solve simple control tasks without hand-crafted features; the MLP learns useful representations directly from raw state vectors.
